# EchoMind — Late Fusion Multimodal com Whisper large-v3-turbo

Este notebook junta as três etapas finais do projeto:

1. **STT/Whisper large-v3-turbo**: áudio gravado → transcrição textual.
2. **Texto/Ollama**: transcrição → distribuição emocional textual.
3. **Prosódia/WavLM + SVC**: áudio → distribuição emocional vocal.
4. **Late fusion**: combinação probabilística das duas distribuições para obter o **top-N final de emoções**.

Labels finais usadas: `joy`, `sadness`, `surprise`, `anger`, `disgust`, `fear`, `neutral`.

> Nota importante: a fusão é feita por **nome da emoção**, não por índice, para evitar erros quando a ordem interna das classes é diferente entre o modelo textual e o modelo de prosódia.


## 0) Como usar localmente

Este notebook está preparado para correr **localmente no teu PC**, que é o cenário recomendado quando o Ollama está instalado localmente.

Estrutura recomendada de pastas, na mesma pasta onde abres o notebook:

```text
projeto/
├── late_fusion_echomind_LOCAL_ollama_whisper_turbo_batch.ipynb
├── prosody_outputs_v5/
│   └── prosody_final_artifact_v5_SSL_WAVLM_RBF_RECOVERED.joblib
├── echomind_test_audios/
│   ├── audio_01.wav
│   ├── audio_02.mp3
│   └── ...
└── late_fusion_outputs/
```

O notebook faz:

```text
pasta com áudios
↓
Whisper turbo / large-v3-turbo
↓
transcrição
↓
Ollama local
↓
WavLM + SVC/prosódia
↓
late fusion
↓
CSV e JSON com resultados finais
```

Antes de correr:
1. garante que tens o Ollama aberto;
2. garante que o modelo existe localmente, por exemplo: `ollama pull qwen2.5:3b-instruct`;
3. coloca a pasta `prosody_outputs_v5` junto ao notebook, ou ajusta a variável `PROSODY_OUTPUTS_DIR`;
4. coloca os teus áudios em `echomind_test_audios`;
5. corre as células por ordem.


In [22]:
# ============================================================
# 1) Instalação opcional de dependências
# ============================================================
# Em Colab, podes pôr INSTALL_DEPS = True na primeira execução.
# Se já tiveres estas bibliotecas instaladas, deixa False para não perder tempo.

INSTALL_DEPS = False

if INSTALL_DEPS:
    import sys, subprocess
    pkgs = [
        "numpy",
        "pandas",
        "joblib",
        "scikit-learn",
        "librosa",
        "soundfile",
        "torch",
        "transformers",
        "accelerate",
        "openai-whisper",  # inclui o modelo turbo nas versões recentes
        "ffmpeg-python",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
    print("Dependências instaladas.")
else:
    print("Instalação saltada. Se alguma importação falhar, muda INSTALL_DEPS para True e corre outra vez.")


Instalação saltada. Se alguma importação falhar, muda INSTALL_DEPS para True e corre outra vez.


In [23]:
# ============================================================
# 2) Imports e configuração global — versão local/Colab flexível
# ============================================================

from pathlib import Path
from typing import Dict, List, Optional, Tuple
import os
import re
import json
import time
import hashlib
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib
import requests

import torch
import librosa
from transformers import AutoFeatureExtractor, AutoModel

# Ordem canónica usada na fusão final.
# A prosódia pode ter outra ordem interna, mas a fusão é sempre por nome.
CANONICAL_EMOTIONS = ["joy", "sadness", "surprise", "anger", "disgust", "fear", "neutral"]

EMOTION_PT = {
    "joy": "alegria",
    "sadness": "tristeza",
    "surprise": "surpresa",
    "anger": "raiva",
    "disgust": "nojo/repulsa",
    "fear": "medo",
    "neutral": "neutro",
}

# ------------------------------------------------------------
# Caminhos principais
# ------------------------------------------------------------
# Local recomendado:
#   ./prosody_outputs_v5/prosody_final_artifact_v5_SSL_WAVLM_RBF_RECOVERED.joblib
#   ./echomind_test_audios/
#   ./late_fusion_outputs/
#
# Também podes configurar por variáveis de ambiente:
#   PROSODY_OUTPUTS_DIR
#   PROSODY_ARTIFACT_PATH
#   TEST_AUDIO_DIR
#   LATE_FUSION_OUTPUT_DIR

PROJECT_ROOT = Path(os.environ.get("ECHOMIND_ROOT", Path.cwd())).resolve()

def find_dir_upwards(start: Path, dirname: str) -> Optional[Path]:
    candidates = [start] + list(start.parents)
    for base in candidates:
        p = base / dirname
        if p.exists() and p.is_dir():
            return p
    return None

default_prosody_dir = find_dir_upwards(PROJECT_ROOT, "prosody_outputs_v5")
if default_prosody_dir is None:
    # fallback local
    default_prosody_dir = PROJECT_ROOT / "prosody_outputs_v5"

PROSODY_OUTPUTS_DIR = Path(os.environ.get("PROSODY_OUTPUTS_DIR", str(default_prosody_dir))).expanduser().resolve()

PROSODY_ARTIFACT_PATH = Path(
    os.environ.get(
        "PROSODY_ARTIFACT_PATH",
        str(PROSODY_OUTPUTS_DIR / "prosody_final_artifact_v5_SSL_WAVLM_RBF_RECOVERED.joblib")
    )
).expanduser().resolve()

LATE_FUSION_OUTPUT_DIR = Path(
    os.environ.get("LATE_FUSION_OUTPUT_DIR", str(PROJECT_ROOT / "late_fusion_outputs"))
).expanduser().resolve()

TEST_AUDIO_DIR = Path(
    os.environ.get("TEST_AUDIO_DIR", str(PROJECT_ROOT / "echomind_test_audios"))
).expanduser().resolve()

LATE_FUSION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEST_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# Ollama / texto
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
TEXT_MODEL = os.environ.get("OLLAMA_TEXT_MODEL", "qwen2.5:3b-instruct")

# Segundo a configuração final anexada, o melhor modelo global é qwen2.5:3b-instruct + Light_Ensemble.
TEXT_STRATEGY = os.environ.get("OLLAMA_TEXT_STRATEGY", "Light_Ensemble")
TEXT_CONFIDENCE_THRESHOLD = 0.50

# Pesos base da fusão.
# Texto tem peso maior porque o modelo textual teve F1-Macro superior ao modelo de prosódia.
BASE_TEXT_WEIGHT = 0.70
BASE_AUDIO_WEIGHT = 0.30

# STT
# No pacote openai-whisper, "turbo" corresponde ao large-v3-turbo.
WHISPER_MODEL_SIZE = os.environ.get("WHISPER_MODEL_SIZE", "turbo")
WHISPER_LANGUAGE = os.environ.get("WHISPER_LANGUAGE", "pt")

print("Configuração:")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROSODY_OUTPUTS_DIR:", PROSODY_OUTPUTS_DIR)
print("PROSODY_ARTIFACT_PATH:", PROSODY_ARTIFACT_PATH)
print("Artefacto existe?", PROSODY_ARTIFACT_PATH.exists())
print("LATE_FUSION_OUTPUT_DIR:", LATE_FUSION_OUTPUT_DIR)
print("TEST_AUDIO_DIR:", TEST_AUDIO_DIR)
print("OLLAMA_HOST:", OLLAMA_HOST)
print("TEXT_MODEL:", TEXT_MODEL)
print("TEXT_STRATEGY:", TEXT_STRATEGY)
print("WHISPER_MODEL_SIZE:", WHISPER_MODEL_SIZE)


Configuração:
PROJECT_ROOT: /home/tomas/MIA/1ANO/2SEM/CA/CA-25_26/tests/final
PROSODY_OUTPUTS_DIR: /home/tomas/MIA/1ANO/2SEM/CA/CA-25_26/tests/final/prosody_outputs_v5
PROSODY_ARTIFACT_PATH: /home/tomas/MIA/1ANO/2SEM/CA/CA-25_26/tests/final/prosody_outputs_v5/prosody_final_artifact_v5_SSL_WAVLM_RBF_RECOVERED.joblib
Artefacto existe? True
LATE_FUSION_OUTPUT_DIR: /home/tomas/MIA/1ANO/2SEM/CA/CA-25_26/tests/final/late_fusion_outputs
TEST_AUDIO_DIR: /home/tomas/MIA/1ANO/2SEM/CA/CA-25_26/tests/final/echomind_test_audios
OLLAMA_HOST: http://localhost:11434
TEXT_MODEL: qwen2.5:3b-instruct
TEXT_STRATEGY: Light_Ensemble
WHISPER_MODEL_SIZE: turbo


In [24]:
# ============================================================
# 3) Google Drive — opcional
# ============================================================
# Esta versão foi preparada para correr localmente.
# Só tenta montar Drive se estiver mesmo em Colab.

try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    print("Google Drive montado.")
except Exception:
    print("Ambiente local ou Colab sem Drive. A continuar com caminhos locais.")

print("Artefacto de prosódia existe?", PROSODY_ARTIFACT_PATH.exists())
if PROSODY_ARTIFACT_PATH.exists():
    print("Tamanho MB:", round(PROSODY_ARTIFACT_PATH.stat().st_size / 1024 / 1024, 2))
else:
    print("ATENÇÃO: artefacto não encontrado.")
    print("Coloca o ficheiro em:", PROSODY_ARTIFACT_PATH)
    print("Ou altera PROSODY_ARTIFACT_PATH / PROSODY_OUTPUTS_DIR na célula de configuração.")


Ambiente local ou Colab sem Drive. A continuar com caminhos locais.
Artefacto de prosódia existe? True
Tamanho MB: 122.1


In [25]:
# ============================================================
# 4) Utilitários comuns
# ============================================================

def normalize_distribution(
    dist: Dict[str, float],
    emotions: List[str] = CANONICAL_EMOTIONS
) -> Dict[str, float]:
    out = {e: max(0.0, float(dist.get(e, 0.0))) for e in emotions}
    total = sum(out.values())
    if total <= 0:
        return {e: (1.0 if e == "neutral" else 0.0) for e in emotions}
    return {e: out[e] / total for e in emotions}


def ranking_from_distribution(
    dist: Dict[str, float],
    top_n: int = 7
) -> List[Tuple[str, float]]:
    dist = normalize_distribution(dist)
    return sorted(dist.items(), key=lambda x: x[1], reverse=True)[:top_n]


def pretty_ranking(dist: Dict[str, float], top_n: int = 3) -> pd.DataFrame:
    rows = []
    for emo, score in ranking_from_distribution(dist, top_n=top_n):
        rows.append({
            "emotion": emo,
            "emotion_pt": EMOTION_PT.get(emo, emo),
            "probability": round(float(score), 4),
        })
    return pd.DataFrame(rows)


def entropy_confidence(
    dist: Dict[str, float],
    emotions: List[str] = CANONICAL_EMOTIONS
) -> float:
    probs = np.array([float(dist.get(e, 0.0)) for e in emotions], dtype=float)
    probs = probs / (probs.sum() + 1e-12)
    probs = np.clip(probs, 1e-12, 1.0)
    entropy = -np.sum(probs * np.log(probs))
    max_entropy = np.log(len(emotions))
    return float(1.0 - entropy / max_entropy)


print("Utilitários comuns prontos.")


Utilitários comuns prontos.


## 5) Classificador textual com Ollama


In [26]:
# ============================================================
# 5) Funções de texto/Ollama
# Adaptadas do notebook ollama_otimizado_melhorado_250_ambiguous_7b
# ============================================================

TEXT_CACHE_PATH = LATE_FUSION_OUTPUT_DIR / "ollama_late_fusion_cache.json"


def load_text_cache() -> Dict:
    if TEXT_CACHE_PATH.exists():
        with open(TEXT_CACHE_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_text_cache(cache: Dict):
    with open(TEXT_CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


TEXT_CACHE = load_text_cache()


def text_cache_key(strategy: str, model: str, text: str, temperature: float) -> str:
    h = hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]
    return f"{strategy}::{model}::{temperature}::{h}"


def parse_distribution_from_text(raw: str) -> Dict[str, float]:
    raw = str(raw).strip()

    # Tenta extrair JSON
    candidates = re.findall(r"\{.*?\}", raw, flags=re.DOTALL)
    for cand in candidates:
        try:
            obj = json.loads(cand)
            if any(e in obj for e in CANONICAL_EMOTIONS):
                return normalize_distribution({e: obj.get(e, 0.0) for e in CANONICAL_EMOTIONS})
        except Exception:
            pass

    # Fallback por pares emotion: number
    scores = {}
    for emo in CANONICAL_EMOTIONS:
        m = re.search(rf"['\"]?{emo}['\"]?\s*[:=]\s*([0-9]*\.?[0-9]+)", raw, flags=re.IGNORECASE)
        if m:
            scores[emo] = float(m.group(1))
    if scores:
        return normalize_distribution(scores)

    # Último fallback: procurar label dominante no texto
    lower = raw.lower()
    for emo in CANONICAL_EMOTIONS:
        if emo in lower:
            return {e: (1.0 if e == emo else 0.0) for e in CANONICAL_EMOTIONS}

    return {e: (1.0 if e == "neutral" else 0.0) for e in CANONICAL_EMOTIONS}


def call_ollama(
    prompt: str,
    model_name: str = TEXT_MODEL,
    temperature: float = 0.1,
    timeout: int = 180
) -> str:
    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": 350,
        },
    }
    r = requests.post(f"{OLLAMA_HOST}/api/generate", json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json().get("response", "")


def check_ollama() -> bool:
    try:
        r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
        r.raise_for_status()
        print("Ollama acessível.")
        models = [m.get("name") for m in r.json().get("models", [])]
        print("Modelos disponíveis:", models[:10])
        return True
    except Exception as e:
        print("Ollama não está acessível neste kernel.")
        print("Erro:", repr(e))
        print("Confirma se o Ollama está ligado e se OLLAMA_HOST está correto.")
        return False


FEW_SHOT_EXAMPLES = [
    ("Hoje recebi uma boa notícia e passei o dia animado e satisfeito.", "joy"),
    ("Senti uma perda enorme e fiquei sem vontade de fazer nada.", "sadness"),
    ("Algo completamente inesperado aconteceu e fiquei sem reação.", "surprise"),
    ("Fiquei irritado com uma injustiça e tive vontade de reclamar.", "anger"),
    ("A situação foi repulsiva, desagradável e deixou-me enojado.", "disgust"),
    ("Estava assustado, ansioso e com medo do que podia acontecer.", "fear"),
    ("O dia foi normal, sem acontecimentos emocionalmente relevantes.", "neutral"),
]


def few_shot_block() -> str:
    lines = []
    for text, label in FEW_SHOT_EXAMPLES:
        dist = {e: (0.86 if e == label else 0.14 / (len(CANONICAL_EMOTIONS) - 1)) for e in CANONICAL_EMOTIONS}
        lines.append(f"Texto: {text}\nResposta: {json.dumps(dist, ensure_ascii=False)}")
    return "\n\n".join(lines)


def query_advanced_v2(text: str, model_name: str = TEXT_MODEL) -> Dict[str, float]:
    prompt = f"""
És um classificador de emoções para relatos diários em português europeu.

Objetivo: devolver uma distribuição probabilística pelas 7 classes:
{', '.join(CANONICAL_EMOTIONS)}.

Critérios principais:
- joy: alegria, satisfação, entusiasmo, orgulho, alívio positivo.
- sadness: perda, vazio, desânimo, choro, sofrimento emocional.
- surprise: acontecimento inesperado, choque, espanto, reação de incredulidade.
- anger: injustiça, frustração, irritação, revolta, traição.
- disgust: nojo, repulsa, desprezo moral ou físico.
- fear: ameaça, ansiedade intensa, insegurança, perigo, pânico.
- neutral: rotina, relato factual, ausência de emoção dominante.

Distingue especialmente:
- fear vs surprise: fear envolve ameaça/ansiedade; surprise envolve inesperado/espanto.
- anger vs disgust: anger envolve revolta/frustração; disgust envolve repulsa/nojo.
- sadness vs neutral: sadness tem sofrimento/perda; neutral é factual/rotineiro.

Exemplos:
{few_shot_block()}

Agora classifica o texto seguinte.
Responde apenas com JSON válido, sem explicações.

Texto:
{text}

JSON:
""".strip()

    raw = call_ollama(prompt, model_name=model_name, temperature=0.1)
    return parse_distribution_from_text(raw)


def query_structured_prompt(text: str, model_name: str = TEXT_MODEL) -> Dict[str, float]:
    prompt = f"""
Tarefa: classificação emocional de um relato diário transcrito.

Analisa internamente os seguintes aspetos, mas NÃO escrevas a análise:
1. Evento principal do relato.
2. Valência emocional: positiva, negativa ou neutra.
3. Presença de ameaça, perda, injustiça, repulsa ou surpresa.
4. Emoção dominante entre: {', '.join(CANONICAL_EMOTIONS)}.

Formato obrigatório da resposta:
{{"joy": 0.0, "sadness": 0.0, "surprise": 0.0, "anger": 0.0, "disgust": 0.0, "fear": 0.0, "neutral": 0.0}}

Texto:
{text}

Resposta JSON:
""".strip()

    raw = call_ollama(prompt, model_name=model_name, temperature=0.1)
    return parse_distribution_from_text(raw)


def post_process_distribution(dist: Dict[str, float], text: str) -> Dict[str, float]:
    dist = normalize_distribution(dist)
    t = text.lower()

    surprise_markers = ["surpresa", "inesperad", "não estava à espera", "nem acreditei", "sem reação", "choque"]
    fear_markers = ["medo", "assust", "ameaç", "pânico", "ansios", "perigo", "tremia", "aperto no peito"]
    anger_markers = ["furioso", "revolt", "irrit", "enganado", "injust", "raiva", "falta de respeito"]
    disgust_markers = ["nojo", "nojento", "repulsa", "enoj", "nauseabundo", "vomit"]

    def boost(label: str, amount: float):
        dist[label] = dist.get(label, 0.0) + amount

    if any(m in t for m in surprise_markers):
        boost("surprise", 0.08)
    if any(m in t for m in fear_markers):
        boost("fear", 0.08)
    if any(m in t for m in anger_markers):
        boost("anger", 0.06)
    if any(m in t for m in disgust_markers):
        boost("disgust", 0.06)

    return normalize_distribution(dist)


def query_advanced_v2_postprocessed(text: str, model_name: str = TEXT_MODEL) -> Dict[str, float]:
    base = query_advanced_v2(text, model_name)
    return post_process_distribution(base, text)


def query_light_ensemble(text: str, model_name: str = TEXT_MODEL) -> Dict[str, float]:
    d1 = query_advanced_v2(text, model_name)
    d2 = query_structured_prompt(text, model_name)
    combined = {e: 0.70 * d1.get(e, 0.0) + 0.30 * d2.get(e, 0.0) for e in CANONICAL_EMOTIONS}
    return normalize_distribution(post_process_distribution(combined, text))


TEXT_STRATEGIES = {
    "Advanced_V2": query_advanced_v2,
    "Advanced_V2_PostProcessed": query_advanced_v2_postprocessed,
    "Light_Ensemble": query_light_ensemble,
}


def query_cached_text(
    strategy_name: str,
    text: str,
    model_name: str = TEXT_MODEL,
    temperature: float = 0.1
) -> Dict[str, float]:
    key = text_cache_key(strategy_name, model_name, text, temperature)
    if key in TEXT_CACHE:
        return normalize_distribution(TEXT_CACHE[key])

    if strategy_name not in TEXT_STRATEGIES:
        raise ValueError(f"Estratégia textual desconhecida: {strategy_name}. Opções: {list(TEXT_STRATEGIES)}")

    dist = TEXT_STRATEGIES[strategy_name](text, model_name)
    dist = normalize_distribution(dist)
    TEXT_CACHE[key] = dist
    save_text_cache(TEXT_CACHE)
    return dist


def classify_text_emotion(
    text: str,
    model_name: str = TEXT_MODEL,
    strategy: str = TEXT_STRATEGY,
    confidence_threshold: float = TEXT_CONFIDENCE_THRESHOLD,
) -> Dict:
    dist = query_cached_text(strategy, text, model_name=model_name)
    emotion = max(dist, key=dist.get)
    confidence = float(dist[emotion])

    return {
        "emotion": emotion,
        "emotion_pt": EMOTION_PT.get(emotion, emotion),
        "confidence": confidence,
        "distribution": dist,
        "ranking": ranking_from_distribution(dist),
        "needs_review": confidence < confidence_threshold,
        "model_name": model_name,
        "strategy": strategy,
    }


print("Classificador textual pronto.")
print("Estratégia textual:", TEXT_STRATEGY)


Classificador textual pronto.
Estratégia textual: Light_Ensemble


In [27]:
# ============================================================
# 6) Teste rápido do Ollama e da componente textual
# ============================================================

_ = check_ollama()

sample_text = (
    "Queria só partilhar que hoje foi um daqueles dias simples, mas mesmo bons. "
    "Acordei bem, as coisas foram fluindo no seu ritmo e senti-me grato."
)

try:
    text_test = classify_text_emotion(sample_text)
    print("Resultado textual:")
    print(json.dumps(text_test, ensure_ascii=False, indent=2))
    display(pretty_ranking(text_test["distribution"], top_n=7))
except Exception as e:
    print("Teste textual falhou.")
    print("Isto normalmente significa que o Ollama não está acessível a partir deste kernel.")
    print("Erro:", repr(e))


Ollama acessível.
Modelos disponíveis: ['qwen2.5:7b-instruct', 'qwen2.5:3b-instruct', 'llama3.2:1b']
Resultado textual:
{
  "emotion": "joy",
  "emotion_pt": "alegria",
  "confidence": 0.9020000000000001,
  "distribution": {
    "joy": 0.9020000000000001,
    "sadness": 0.01633333333333334,
    "surprise": 0.01633333333333334,
    "anger": 0.01633333333333334,
    "disgust": 0.01633333333333334,
    "fear": 0.01633333333333334,
    "neutral": 0.01633333333333334
  },
  "ranking": [
    [
      "joy",
      0.9020000000000001
    ],
    [
      "sadness",
      0.01633333333333334
    ],
    [
      "surprise",
      0.01633333333333334
    ],
    [
      "anger",
      0.01633333333333334
    ],
    [
      "disgust",
      0.01633333333333334
    ],
    [
      "fear",
      0.01633333333333334
    ],
    [
      "neutral",
      0.01633333333333334
    ]
  ],
  "needs_review": false,
  "model_name": "qwen2.5:3b-instruct",
  "strategy": "Light_Ensemble"
}


,emotion,emotion_pt,probability
0,joy,alegria,0.9020
1,sadness,tristeza,0.0163
2,surprise,surpresa,0.0163
3,anger,raiva,0.0163
4,disgust,nojo/repulsa,0.0163
5,fear,medo,0.0163
6,neutral,neutro,0.0163


## 7) Classificador de prosódia com WavLM + artefacto final


In [28]:
# ============================================================
# 7) Funções de prosódia/WavLM
# ============================================================

SSL_MAX_SECONDS_PER_CHUNK = 12.0
SSL_CHUNK_OVERLAP_SECONDS = 1.0

_SSL_CACHE = {}


def safe_load_audio(path, sr=16000):
    y, sr = librosa.load(str(path), sr=sr, mono=True)
    y = np.asarray(y, dtype=np.float32)
    y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
    if len(y) == 0:
        raise ValueError(f"Áudio vazio: {path}")
    return y, sr


def trim_silence(y, top_db=30):
    try:
        yt, _ = librosa.effects.trim(y, top_db=top_db)
        return yt.astype(np.float32) if len(yt) else y.astype(np.float32)
    except Exception:
        return y.astype(np.float32)


def iter_chunks(y, sr, max_seconds=SSL_MAX_SECONDS_PER_CHUNK, overlap_seconds=SSL_CHUNK_OVERLAP_SECONDS):
    max_len = int(max_seconds * sr)
    overlap = int(overlap_seconds * sr)

    if len(y) <= max_len:
        yield y
        return

    step = max(1, max_len - overlap)
    start = 0

    while start < len(y):
        end = min(len(y), start + max_len)
        chunk = y[start:end]
        if len(chunk) > int(0.25 * sr):
            yield chunk
        if end >= len(y):
            break
        start += step


def device_name():
    return "cuda" if torch.cuda.is_available() else "cpu"


def load_ssl_model(model_name="microsoft/wavlm-base"):
    if model_name in _SSL_CACHE:
        return _SSL_CACHE[model_name]

    device = device_name()
    print("A carregar modelo SSL:", model_name, "em", device)

    feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    _SSL_CACHE[model_name] = (feature_extractor, model, device)
    return _SSL_CACHE[model_name]


def extract_ssl_embedding(
    audio_path,
    model_name="microsoft/wavlm-base",
    target_sr=16000
) -> np.ndarray:
    feature_extractor, model, device = load_ssl_model(model_name)

    y, sr = safe_load_audio(audio_path, sr=target_sr)
    y = trim_silence(y)

    embs = []

    for chunk in iter_chunks(y, sr):
        inputs = feature_extractor(chunk, sampling_rate=sr, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            out = model(**inputs)
            h = out.last_hidden_state.squeeze(0)

            mean = h.mean(dim=0).detach().cpu().numpy()
            std = h.std(dim=0).detach().cpu().numpy()

        # WavLM base: 768 mean + 768 std = 1536 dims
        embs.append(np.concatenate([mean, std]).astype(np.float32))

    if not embs:
        raise RuntimeError("Não foi possível extrair embedding SSL.")

    return np.vstack(embs).mean(axis=0).astype(np.float32)


def load_prosody_artifact(path: Path = PROSODY_ARTIFACT_PATH) -> Dict:
    if not Path(path).exists():
        raise FileNotFoundError(f"Artefacto de prosódia não encontrado: {path}")

    art = joblib.load(path)

    print("Artefacto de prosódia carregado.")
    print("Feature set:", art.get("feature_set") or art.get("selected_strategy"))
    print("Modelo:", art.get("model_name") or art.get("final_model_name"))
    print("Emoções do artefacto:", art.get("emotions"))
    print("Dimensão SSL:", art.get("ssl_embedding_dim", "desconhecida"))

    return art


PROSODY_ARTIFACT = load_prosody_artifact(PROSODY_ARTIFACT_PATH)


def _get_model_classes(model):
    if hasattr(model, "classes_"):
        return model.classes_
    if hasattr(model, "named_steps") and "clf" in model.named_steps and hasattr(model.named_steps["clf"], "classes_"):
        return model.named_steps["clf"].classes_
    return None


def classify_prosody_audio(
    audio_path,
    artifact: Dict = PROSODY_ARTIFACT,
) -> Dict:
    model = artifact.get("model") or artifact.get("final_model")
    if model is None:
        raise RuntimeError("O artefacto não contém 'model' nem 'final_model'.")

    ssl_model_name = artifact.get("ssl_model_name", "microsoft/wavlm-base")
    ssl_sample_rate = int(artifact.get("ssl_sample_rate", 16000))

    emb = extract_ssl_embedding(
        audio_path,
        model_name=ssl_model_name,
        target_sr=ssl_sample_rate,
    ).reshape(1, -1)

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(emb)[0]
    else:
        raise RuntimeError("O modelo de prosódia não suporta predict_proba.")

    idx_to_label = artifact.get("idx_to_label", {})
    # joblib preserva ints, mas deixamos robusto para JSON/string keys
    idx_to_label = {int(k): v for k, v in idx_to_label.items()}

    artifact_emotions = artifact.get("emotions", CANONICAL_EMOTIONS)
    classes = _get_model_classes(model)

    dist = {e: 0.0 for e in CANONICAL_EMOTIONS}

    if classes is not None:
        for cls, p in zip(classes, proba):
            cls_int = int(cls)
            emo = idx_to_label.get(cls_int)
            if emo is None and 0 <= cls_int < len(artifact_emotions):
                emo = artifact_emotions[cls_int]
            if emo in dist:
                dist[emo] += float(p)
    else:
        # fallback: assume ordem do artefacto, nunca a ordem canónica
        for i, p in enumerate(proba):
            if i < len(artifact_emotions):
                emo = artifact_emotions[i]
                if emo in dist:
                    dist[emo] += float(p)

    dist = normalize_distribution(dist)
    emotion = max(dist, key=dist.get)
    confidence = float(dist[emotion])

    return {
        "emotion": emotion,
        "emotion_pt": EMOTION_PT.get(emotion, emotion),
        "confidence": confidence,
        "distribution": dist,
        "ranking": ranking_from_distribution(dist),
        "strategy": artifact.get("feature_set") or artifact.get("selected_strategy"),
        "model_name": artifact.get("model_name") or artifact.get("final_model_name"),
        "audio_path": str(audio_path),
    }


print("Classificador de prosódia pronto.")


Artefacto de prosódia carregado.
Feature set: ssl_wavlm
Modelo: rbf_svc_balanced
Emoções do artefacto: ['joy', 'sadness', 'surprise', 'fear', 'anger', 'disgust', 'neutral']
Dimensão SSL: 1536
Classificador de prosódia pronto.


## 8) STT — transcrição automática do áudio


In [29]:
# ============================================================
# 8) STT com Whisper
# ============================================================

_WHISPER_CACHE = {}


def load_whisper_model(model_size: str = WHISPER_MODEL_SIZE):
    if model_size in _WHISPER_CACHE:
        return _WHISPER_CACHE[model_size]

    import whisper

    print("A carregar Whisper:", model_size)
    try:
        model = whisper.load_model(model_size)
    except Exception as e:
        if model_size in {"large-v3-turbo", "whisper-large-v3-turbo"}:
            print("Não carregou com esse nome. A tentar alias 'turbo'...")
            model = whisper.load_model("turbo")
        else:
            raise e
    _WHISPER_CACHE[model_size] = model
    return model


def transcribe_audio_whisper(
    audio_path,
    model_size: str = WHISPER_MODEL_SIZE,
    language: str = WHISPER_LANGUAGE,
) -> Dict:
    model = load_whisper_model(model_size)

    result = model.transcribe(
        str(audio_path),
        language=language,
        task="transcribe",
        fp16=torch.cuda.is_available(),
        verbose=False,
    )

    text = str(result.get("text", "")).strip()

    return {
        "text": text,
        "language": language,
        "model_size": model_size,
        "raw": result,
    }


print("Funções de STT prontas.")


Funções de STT prontas.


## 9) Late fusion


In [30]:
# ============================================================
# 9) Late fusion texto + prosódia
# ============================================================

def fuse_text_audio_emotions(
    text_distribution: Dict[str, float],
    audio_distribution: Dict[str, float],
    base_text_weight: float = BASE_TEXT_WEIGHT,
    base_audio_weight: float = BASE_AUDIO_WEIGHT,
    adaptive: bool = True,
    min_text_weight: float = 0.55,
    max_text_weight: float = 0.85,
) -> Dict:
    text_distribution = normalize_distribution(text_distribution)
    audio_distribution = normalize_distribution(audio_distribution)

    text_weight = float(base_text_weight)
    audio_weight = float(base_audio_weight)

    if adaptive:
        text_conf = entropy_confidence(text_distribution)
        audio_conf = entropy_confidence(audio_distribution)

        # Ajuste interpretável:
        # - mantém o texto como modalidade principal;
        # - aumenta ligeiramente o peso da modalidade mais confiante.
        text_weight = base_text_weight * (0.75 + 0.50 * text_conf)
        audio_weight = base_audio_weight * (0.75 + 0.50 * audio_conf)

        total = text_weight + audio_weight
        text_weight = text_weight / (total + 1e-12)
        audio_weight = audio_weight / (total + 1e-12)

        text_weight = min(max(text_weight, min_text_weight), max_text_weight)
        audio_weight = 1.0 - text_weight

    final_dist = {}
    for emo in CANONICAL_EMOTIONS:
        final_dist[emo] = (
            text_weight * float(text_distribution.get(emo, 0.0)) +
            audio_weight * float(audio_distribution.get(emo, 0.0))
        )

    final_dist = normalize_distribution(final_dist)
    top_emotion = max(final_dist, key=final_dist.get)

    return {
        "emotion": top_emotion,
        "emotion_pt": EMOTION_PT.get(top_emotion, top_emotion),
        "confidence": float(final_dist[top_emotion]),
        "distribution": final_dist,
        "ranking": ranking_from_distribution(final_dist),
        "weights": {
            "text": float(text_weight),
            "audio": float(audio_weight),
        },
        "entropy_confidence": {
            "text": entropy_confidence(text_distribution),
            "audio": entropy_confidence(audio_distribution),
            "final": entropy_confidence(final_dist),
        },
    }


def predict_emotion_from_audio(
    audio_path,
    manual_transcript: Optional[str] = None,
    use_stt: bool = True,
    text_model: str = TEXT_MODEL,
    text_strategy: str = TEXT_STRATEGY,
    adaptive_fusion: bool = True,
    top_n: int = 3,
) -> Dict:
    audio_path = Path(audio_path)

    if not audio_path.exists():
        raise FileNotFoundError(f"Áudio não encontrado: {audio_path}")

    # 1) STT / transcrição
    if manual_transcript is not None and str(manual_transcript).strip():
        transcript = str(manual_transcript).strip()
        stt_result = {
            "text": transcript,
            "source": "manual_transcript",
        }
    elif use_stt:
        stt_result = transcribe_audio_whisper(audio_path)
        transcript = stt_result["text"]
        stt_result["source"] = "whisper"
    else:
        raise ValueError("Sem manual_transcript e use_stt=False. Não há texto para o Ollama.")

    if not transcript:
        raise RuntimeError("A transcrição ficou vazia.")

    # 2) Emoção textual
    text_result = classify_text_emotion(
        transcript,
        model_name=text_model,
        strategy=text_strategy,
    )

    # 3) Emoção prosódica
    audio_result = classify_prosody_audio(audio_path)

    # 4) Late fusion
    fusion_result = fuse_text_audio_emotions(
        text_result["distribution"],
        audio_result["distribution"],
        adaptive=adaptive_fusion,
    )

    result = {
        "audio_path": str(audio_path),
        "transcript": transcript,
        "stt": {
            "source": stt_result.get("source"),
            "language": stt_result.get("language"),
            "model_size": stt_result.get("model_size"),
        },
        "text": text_result,
        "audio": audio_result,
        "fusion": fusion_result,
        "top_n": pretty_ranking(fusion_result["distribution"], top_n=top_n).to_dict(orient="records"),
    }

    return result


def display_prediction_result(result: Dict, top_n: int = 7):
    print("Áudio:", result["audio_path"])
    print("\nTranscrição:")
    print(result["transcript"])

    print("\nTexto/Ollama:")
    print(result["text"]["emotion"], "| confiança:", round(result["text"]["confidence"], 4))

    print("\nProsódia/WavLM:")
    print(result["audio"]["emotion"], "| confiança:", round(result["audio"]["confidence"], 4))

    print("\nLate fusion final:")
    print(result["fusion"]["emotion"], "| confiança:", round(result["fusion"]["confidence"], 4))
    print("Pesos:", result["fusion"]["weights"])

    print("\nTop emoções finais:")
    display(pretty_ranking(result["fusion"]["distribution"], top_n=top_n))


print("Funções de late fusion prontas.")


Funções de late fusion prontas.


## 10) Testar num áudio


In [31]:
# ============================================================
# 10) Teste num áudio
# ============================================================
# Opção A: usar STT automático
# audio_path = "/content/drive/MyDrive/echomind_test_audios/happy_01.wav"
# result = predict_emotion_from_audio(audio_path, use_stt=True)
# display_prediction_result(result)

# Opção B: usar transcrição manual, útil se o Whisper ainda não estiver instalado
# ou se quiseres testar rapidamente a lógica da fusão.
audio_path = None  # muda para o caminho do teu áudio, por exemplo: "/content/drive/MyDrive/echomind_test_audios/happy_01.wav"

manual_transcript = (
    "Queria só partilhar que hoje foi um daqueles dias simples, mas mesmo bons. "
    "Não aconteceu nada de especial, mas acordei bem, as coisas foram fluindo "
    "e fui encontrando pequenos momentos que me deixaram com um sorriso. "
    "Hoje foi simples, tranquilo, e estou grato por isso."
)

if audio_path is not None:
    result = predict_emotion_from_audio(
        audio_path,
        manual_transcript=None,  # None para usar Whisper large-v3-turbo
        use_stt=True,        # True para transcrever automaticamente
        top_n=7,
    )

    display_prediction_result(result, top_n=7)

    # Guardar resultado individual
    out_json = LATE_FUSION_OUTPUT_DIR / f"late_fusion_result_{Path(audio_path).stem}.json"
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2, default=str)

    print("\nResultado guardado em:", out_json)
else:
    print("Define audio_path para testares um áudio real.")


Define audio_path para testares um áudio real.


## 11) Testar vários áudios numa pasta


In [32]:
# ============================================================
# 11) Batch prediction numa pasta
# ============================================================

AUDIO_EXTS = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac"}


def predict_folder(
    folder_path,
    use_stt: bool = True,
    manual_transcripts: Optional[Dict[str, str]] = None,
    max_files: Optional[int] = None,
) -> pd.DataFrame:
    folder_path = Path(folder_path)

    if not folder_path.exists():
        raise FileNotFoundError(f"Pasta não encontrada: {folder_path}")

    files = [p for p in sorted(folder_path.rglob("*")) if p.suffix.lower() in AUDIO_EXTS]

    if max_files is not None:
        files = files[:max_files]

    rows = []
    all_results = []

    for audio_path in files:
        print("\n" + "=" * 80)
        print("A processar:", audio_path)

        manual_text = None
        if manual_transcripts:
            manual_text = manual_transcripts.get(audio_path.name) or manual_transcripts.get(str(audio_path))

        try:
            res = predict_emotion_from_audio(
                audio_path,
                manual_transcript=manual_text,
                use_stt=use_stt if manual_text is None else False,
                top_n=7,
            )

            all_results.append(res)

            rows.append({
                "audio_path": str(audio_path),
                "filename": audio_path.name,
                "transcript": res["transcript"],
                "text_emotion": res["text"]["emotion"],
                "text_confidence": res["text"]["confidence"],
                "audio_emotion": res["audio"]["emotion"],
                "audio_confidence": res["audio"]["confidence"],
                "final_emotion": res["fusion"]["emotion"],
                "final_emotion_pt": res["fusion"]["emotion_pt"],
                "final_confidence": res["fusion"]["confidence"],
                "text_weight": res["fusion"]["weights"]["text"],
                "audio_weight": res["fusion"]["weights"]["audio"],
                "top3": "; ".join([f"{x['emotion']}={x['probability']}" for x in res["top_n"][:3]]),
            })

            print("Final:", res["fusion"]["emotion"], "|", round(res["fusion"]["confidence"], 4))

        except Exception as e:
            print("ERRO:", repr(e))
            rows.append({
                "audio_path": str(audio_path),
                "filename": audio_path.name,
                "error": repr(e),
            })

    df = pd.DataFrame(rows)

    ts = time.strftime("%Y%m%d_%H%M%S")
    out_csv = LATE_FUSION_OUTPUT_DIR / f"late_fusion_batch_results_{ts}.csv"
    out_json = LATE_FUSION_OUTPUT_DIR / f"late_fusion_batch_results_{ts}.json"

    df.to_csv(out_csv, index=False)

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2, default=str)

    print("\nCSV guardado em:", out_csv)
    print("JSON guardado em:", out_json)

    return df


# Exemplo:
# df_batch = predict_folder(TEST_AUDIO_DIR, use_stt=True, max_files=5)
# display(df_batch)


In [33]:
# ============================================================
# 12) EXECUÇÃO FINAL — processar todos os áudios da pasta
# ============================================================
# 1) Mete os teus ficheiros .wav/.mp3/.m4a/etc. na pasta:
#    TEST_AUDIO_DIR
#
# 2) Depois corre esta célula. Ela faz, para cada áudio:
#    áudio -> Whisper turbo -> texto -> Ollama -> prosódia -> late fusion -> CSV/JSON final

print("Pasta de áudios:", TEST_AUDIO_DIR)
print("Existe?", TEST_AUDIO_DIR.exists())

audio_files = []
if TEST_AUDIO_DIR.exists():
    audio_files = [p for p in sorted(TEST_AUDIO_DIR.rglob("*")) if p.suffix.lower() in AUDIO_EXTS]

print("Nº de áudios encontrados:", len(audio_files))
for p in audio_files[:10]:
    print("-", p.name)

if TEST_AUDIO_DIR.exists() and audio_files:
    df_batch = predict_folder(TEST_AUDIO_DIR, use_stt=True, max_files=None)
    display(df_batch)
else:
    print("Coloca os teus áudios em TEST_AUDIO_DIR antes de correr esta célula.")


Pasta de áudios: /home/tomas/MIA/1ANO/2SEM/CA/CA-25_26/tests/final/echomind_test_audios
Existe? True
Nº de áudios encontrados: 7
- anger.wav
- disgust.wav
- fear.wav
- joy.wav
- neutral.wav
- sad.wav
- surprise.wav

A processar: /home/tomas/MIA/1ANO/2SEM/CA/CA-25_26/tests/final/echomind_test_audios/anger.wav
A carregar Whisper: turbo


100%|█████████████████████████████████████| 1.51G/1.51G [05:09<00:00, 5.22MiB/s]
100%|██████████| 2074/2074 [00:09<00:00, 228.81frames/s]


A carregar modelo SSL: microsoft/wavlm-base em cpu


Loading weights: 100%|██████████| 248/248 [00:00<00:00, 51458.76it/s]


Final: sadness | 0.5048

A processar: /home/tomas/MIA/1ANO/2SEM/CA/CA-25_26/tests/final/echomind_test_audios/disgust.wav


...

100%|██████████| 2624/2624 [00:13<00:00, 195.83frames/s]


Final: sadness | 0.763

A processar: /home/tomas/MIA/1ANO/2SEM/CA/CA-25_26/tests/final/echomind_test_audios/surprise.wav


100%|██████████| 2124/2124 [00:13<00:00, 163.24frames/s]


Final: surprise | 0.5938

CSV guardado em: /home/tomas/MIA/1ANO/2SEM/CA/CA-25_26/tests/final/late_fusion_outputs/late_fusion_batch_results_20260525_164349.csv
JSON guardado em: /home/tomas/MIA/1ANO/2SEM/CA/CA-25_26/tests/final/late_fusion_outputs/late_fusion_batch_results_20260525_164349.json


,audio_path,filename,transcript,text_emotion,text_confidence,audio_emotion,audio_confidence,final_emotion,final_emotion_pt,final_confidence,text_weight,audio_weight,top3
0,/home/tomas/MIA/1ANO/2SEM/CA/CA-25_26/tests/fi...,anger.wav,Hoje fiquei bastante irritado Houve uma situaç...,sadness,0.575558,sadness,0.308780,sadness,tristeza,0.504840,0.734916,0.265084,sadness=0.5048; anger=0.3074; joy=0.0735
1,/home/tomas/MIA/1ANO/2SEM/CA/CA-25_26/tests/fi...,disgust.wav,Hoje houve uma situação que me deixou mesmo in...,disgust,0.858929,joy,0.353455,disgust,nojo/repulsa,0.646734,0.744717,0.255283,disgust=0.6467; joy=0.1011; sadness=0.0961
2,/home/tomas/MIA/1ANO/2SEM/CA/CA-25_26/tests/fi...,fear.wav,Hoje passei o dia com bastante ansiedade. Desd...,sadness,0.602000,sadness,0.579531,sadness,tristeza,0.595549,0.712899,0.287101,sadness=0.5955; fear=0.2672; joy=0.0616
3,/home/tomas/MIA/1ANO/2SEM/CA/CA-25_26/tests/fi...,joy.wav,Hoje foi um dia muito bom. Acordei com energia...,joy,0.975862,joy,0.431341,joy,alegria,0.847761,0.764746,0.235254,joy=0.8478; neutral=0.0545; sadness=0.0408
4,/home/tomas/MIA/1ANO/2SEM/CA/CA-25_26/tests/fi...,neutral.wav,"Hoje foi um dia normal. Acordei, fiz as minhas...",neutral,0.908125,joy,0.501908,neutral,neutro,0.694209,0.744265,0.255735,neutral=0.6942; joy=0.1398; sadness=0.0786
5,/home/tomas/MIA/1ANO/2SEM/CA/CA-25_26/tests/fi...,sad.wav,Hoje foi um dia pesado. Acordei sem grande von...,sadness,0.902000,sadness,0.347620,sadness,tristeza,0.763001,0.749270,0.250730,sadness=0.763; joy=0.0942; neutral=0.0594
6,/home/tomas/MIA/1ANO/2SEM/CA/CA-25_26/tests/fi...,surprise.wav,O meu dia começou de forma completamente norma...,surprise,0.816667,joy,0.613815,surprise,surpresa,0.593806,0.726542,0.273458,surprise=0.5938; joy=0.1788; neutral=0.0966


## 12) Notas para defesa/apresentação

- O sistema processa uma pasta de áudios localmente.
- A transcrição usa Whisper `large-v3-turbo` através do alias `turbo` do pacote `openai-whisper`.
- A componente textual usa `qwen2.5:3b-instruct` no Ollama local com estratégia `Light_Ensemble`.
- A componente de prosódia usa `WavLM base` para extrair embeddings de 1536 dimensões e um `SVC RBF` com `probability=True`.
- A fusão é feita ao nível das probabilidades (**late fusion**), não ao nível das labels finais.
- O peso base favorece o texto (`70%`) porque o modelo textual apresentou desempenho superior ao modelo de prosódia.
- O peso pode ser ajustado de forma adaptativa com base na entropia/confiança das distribuições.
- A fusão é feita por nome da emoção para evitar erros quando a ordem interna das classes é diferente entre modelos.
